# Imports

In [ ]:
import pathlib
import pandas as pd
import numpy as np

from experiments.models import (
    LightGBMForecast,
    LightGBMARForecast,
)

from experiments.utils import (
    load_experiments_specs,
    create_tsfeatures,
)

# Load Experiment Specifications

In [ ]:
dataset = "ABC"

In [ ]:
# Load specifications
train_type = "global"
experiment_specs = load_experiments_specs(
    dataset=dataset,
    train_type=train_type,
)

# Train and Test Data
train_df = experiment_specs["train"]
test_df = experiment_specs["test"]

# Meta Data
meta = experiment_specs["meta"]
features = meta["features"]
fcst_h = meta["fcst_h"]
freq = meta["freq"]
lags = meta["lags"]
max_lag = meta["max_lag"]
n_series = meta["n_series"]
seed = 123

# Hyper-Parameters
config = experiment_specs["config"]
params_lgb = config["lgb_params"]

# TS-Features

In [ ]:
# Extract time series features
ts_feats_df, ts_feats = create_tsfeatures(
    train=train_df,
    freq=freq
)

# Add features to train and test
train_df = pd.merge(train_df, ts_feats_df, on="series_id", how="left")
test_df = pd.merge(test_df, ts_feats_df, on="series_id", how="left")
features = features + ts_feats

# LightGBM

In [ ]:
np.random.seed(seed)

if params_lgb["use_time_index"]:
    lgb_feats = meta["features"] + ["time_idx"] + ts_feats
else:
    lgb_feats = meta["features"] + ts_feats

lgb_fcst = LightGBMForecast(
    params_lgb,
    train_df,
    test_df.drop(columns=["value"]),
    lgb_feats
)

# LightGBM-AR

In [ ]:
np.random.seed(seed)

lgb_ar_fcst = LightGBMARForecast(
    params_lgb,
    train_df,
    test_df.drop(columns=["value"]),
    features,
    freq,
    fcst_h,
    lags,
    max_lag,
    n_series
)

# Combine Forecasts

In [ ]:
# Concatenate all forecasts into a single DataFrame
fcsts_df = pd.concat(
    [
        lgb_fcst,
        lgb_ar_fcst,
    ],
    axis=0
)

# Add actual values and dataset information
fcsts_df = pd.merge(
    fcsts_df,
    test_df[["dataset", "series_id", "date", "value"]],
    on=["series_id", "date"],
    how="inner"
)

# Save Forecasts

In [ ]:
repo_root = pathlib.Path.cwd()
while not (repo_root / "pyproject.toml").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

result_path = repo_root / "experiments" / "runs" / "results" / train_type
result_path.mkdir(parents=True, exist_ok=True)

fcsts_df.to_csv(result_path / f"{dataset.lower()}_lgbm_fcsts.csv", index=False)